# DyTopo — Mock Demo & Topology Evolution

**Paper:** Lu et al. 2026 — *DyTopo: Dynamic Topology Routing for Multi-Agent Reasoning via Semantic Matching* (arXiv:2602.06039)

## About the dataset

The `dytopo_engine.py` backend uses **hand-crafted mock responses** (no real dataset) stored in `_CODE_MOCK` and `_MATH_MOCK` dicts.
Each entry is keyed by `(role, round_t)` and provides:
- `public_content` — the agent's visible output
- `q_vector` — a natural-language *query descriptor* (what this agent needs)
- `k_vector` — a natural-language *key descriptor* (what this agent provides)

The Q/K strings were written so that when encoded by `all-MiniLM-L6-v2` and compared via cosine similarity at τ=0.3, they naturally produce the paper's **Figure 4 topology evolution**:
- **Round 1**: broad, exploratory Q/K → high cosine overlap → **dense graph**
- **Round 2**: role-specific, targeted Q/K → moderate overlap → **medium graph**
- **Round 3**: completion-stage Q/K → low overlap → **sparse graph**

This notebook runs Algorithm 1 in mock mode and visualises the adjacency matrices.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'api'))

import asyncio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from matplotlib.gridspec import GridSpec

# Path assumes notebook is at big-data/notebooks/
API_DIR = os.path.join(os.path.dirname(os.path.abspath('')), 'api')
if API_DIR not in sys.path:
    sys.path.insert(0, API_DIR)

print('API dir:', API_DIR)

In [ ]:
from dytopo_engine import DyTopoConfig, run_dytopo, build_adjacency, encode
print('dytopo_engine imported OK')

## 1. Run Algorithm 1 in mock mode (code task)

In [ ]:
PROBLEM = """
Write a Python function that takes a list of integers `nums` and returns the index
of the first duplicate element, or -1 if none exists. Example: nums=[2,7,11,2] → 0.
""".strip()

cfg = DyTopoConfig(
    task_type='code',
    n_agents=4,
    tau=0.3,
    max_rounds=5,
    mock_mode=True,
)

async def collect_rounds():
    results = []
    async for rr in run_dytopo(PROBLEM, cfg):
        results.append(rr)
        print(f'  Round {rr.round} — goal: {rr.goal[:60]}... complete={rr.is_complete}')
    return results

print('Running DyTopo (mock mode)...')
rounds = asyncio.run(collect_rounds())
print(f'\nTotal rounds: {len(rounds)}')

## 2. Adjacency matrices — heat maps

In [ ]:
ROLES = [a.role for a in rounds[0].agents]
N = len(ROLES)

fig, axes = plt.subplots(1, len(rounds), figsize=(4 * len(rounds), 4))
if len(rounds) == 1:
    axes = [axes]

cmap = plt.cm.Blues

for ax, rr in zip(axes, rounds):
    A = np.array(rr.adjacency)
    n_edges = A.sum()
    density = n_edges / (N * (N - 1)) if N > 1 else 0

    im = ax.imshow(A, cmap=cmap, vmin=0, vmax=1, aspect='equal')
    ax.set_xticks(range(N))
    ax.set_yticks(range(N))
    ax.set_xticklabels(ROLES, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ROLES, fontsize=8)
    ax.set_title(f'Round {rr.round + 1}\nedges={n_edges}  density={density:.0%}', fontsize=9)
    ax.set_xlabel('From (source)', fontsize=8)
    ax.set_ylabel('To (destination)', fontsize=8)

    # annotate cells
    for i in range(N):
        for j in range(N):
            ax.text(j, i, str(A[i, j]), ha='center', va='center',
                    color='white' if A[i, j] else 'lightgray', fontsize=11, fontweight='bold')

fig.suptitle('DyTopo Adjacency Matrices — Code Task (Mock)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('dytopo_adjacency_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: dytopo_adjacency_matrices.png')

## 3. Topology as directed graphs (NetworkX)

In [ ]:
ROLE_COLORS = {
    'Developer':  '#38bdf8',
    'Researcher': '#4ade80',
    'Tester':     '#fbbf24',
    'Designer':   '#a78bfa',
}

fig, axes = plt.subplots(1, len(rounds), figsize=(5 * len(rounds), 5))
if len(rounds) == 1:
    axes = [axes]

# Circular layout — same positions across rounds for visual consistency
G_ref = nx.DiGraph()
G_ref.add_nodes_from(range(N))
pos = nx.circular_layout(G_ref)

PHASE_LABELS = ['Exploration', 'Verification', 'Finalization']

for ax, rr in zip(axes, rounds):
    A = np.array(rr.adjacency)
    G = nx.DiGraph()
    G.add_nodes_from(range(N))
    for i in range(N):
        for j in range(N):
            if A[i, j]:  # edge i→j
                G.add_edge(i, j)

    node_colors = [ROLE_COLORS.get(ROLES[i], '#94a3b8') for i in range(N)]
    node_sizes  = [800 + 200 * G.in_degree(i) for i in range(N)]
    labels = {i: ROLES[i] for i in range(N)}

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes, alpha=0.9)
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=8, font_color='white', font_weight='bold')
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#64748b', arrows=True,
                           arrowstyle='-|>', arrowsize=15, connectionstyle='arc3,rad=0.15')

    n_edges = A.sum()
    density = n_edges / (N * (N - 1)) if N > 1 else 0
    phase = PHASE_LABELS[min(rr.round, len(PHASE_LABELS)-1)]
    ax.set_title(f'Round {rr.round + 1} — {phase}\n{n_edges} edges · {density:.0%} density',
                 fontsize=9, pad=10)
    ax.axis('off')
    ax.set_facecolor('#0a0e1a')

fig.patch.set_facecolor('#0a0e1a')
fig.suptitle('DyTopo Topology Evolution — Code Task (Mock)\nPaper Figure 4 reproduction', 
             fontsize=12, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('dytopo_topology_graphs.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: dytopo_topology_graphs.png')

## 4. Edge density evolution

In [ ]:
densities = []
for rr in rounds:
    A = np.array(rr.adjacency)
    n_edges = A.sum()
    densities.append(n_edges / (N * (N - 1)))

fig, ax = plt.subplots(figsize=(7, 4))
rounds_x = [r.round + 1 for r in rounds]

ax.plot(rounds_x, densities, 'o-', color='#fbbf24', linewidth=2.5, markersize=10, markerfacecolor='#fbbf24')
ax.fill_between(rounds_x, densities, alpha=0.15, color='#fbbf24')

PHASE_COLORS = ['#38bdf8', '#fbbf24', '#4ade80']
PHASE_LABELS = ['Exploration', 'Verification', 'Finalization']
for i, (x, d, rr) in enumerate(zip(rounds_x, densities, rounds)):
    phase = PHASE_LABELS[min(i, 2)]
    ax.annotate(f'{d:.0%}\n{phase}', xy=(x, d), xytext=(0, 16), textcoords='offset points',
                ha='center', fontsize=8, color=PHASE_COLORS[min(i, 2)], fontweight='bold')

ax.set_xlabel('Communication Round', fontsize=10)
ax.set_ylabel('Edge Density (active edges / max edges)', fontsize=10)
ax.set_title('DyTopo Edge Density Evolution\n(matches paper Figure 4 — dense exploration → sparse finalization)', fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_xticks(rounds_x)
ax.set_xticklabels([f'Round {x}' for x in rounds_x])
ax.grid(axis='y', alpha=0.3)
ax.set_facecolor('#0f1828')
fig.patch.set_facecolor('#0a0e1a')
ax.tick_params(colors='white')
ax.title.set_color('white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.spines[:].set_color('#1e293b')

plt.tight_layout()
plt.savefig('dytopo_density_evolution.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print('Saved: dytopo_density_evolution.png')

## 5. Agent outputs summary

In [ ]:
import pandas as pd

rows = []
for rr in rounds:
    for a in rr.agents:
        rows.append({
            'Round': rr.round + 1,
            'Role': a.role,
            'In-Neighbors': len(a.in_neighbors),
            'Q-vector (truncated)': a.q_vector[:60] + '...' if len(a.q_vector) > 60 else a.q_vector,
            'K-vector (truncated)': a.k_vector[:60] + '...' if len(a.k_vector) > 60 else a.k_vector,
        })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 80)
print(df.to_string(index=False))

## 6. Run math task — compare topology

Math tasks use different roles (ProblemParser/Solver/Verifier) and are configured with τ=0.4 (per paper Table 3: omni_math optimal at τ=0.4).

In [ ]:
MATH_PROBLEM = """
Find all Pythagorean triples (x, y, z) where x < y < z and z ≤ 20.
A Pythagorean triple satisfies x² + y² = z².
""".strip()

cfg_math = DyTopoConfig(
    task_type='math',
    n_agents=3,  # ProblemParser, Solver, Verifier
    tau=0.4,
    max_rounds=5,
    mock_mode=True,
)

async def collect_math():
    results = []
    async for rr in run_dytopo(MATH_PROBLEM, cfg_math):
        results.append(rr)
        A = np.array(rr.adjacency)
        density = A.sum() / (len(rr.agents) * (len(rr.agents) - 1))
        print(f'  Round {rr.round} — density={density:.0%} complete={rr.is_complete}')
    return results

print('Running DyTopo math task (mock mode)...')
math_rounds = asyncio.run(collect_math())

# Print final answer
for rr in math_rounds:
    if rr.final_answer:
        print(f'\nFinal answer (Round {rr.round}):') 
        print(rr.final_answer)